In [ ]:
# configuração do SparkSession para usar Delta Lake


# ==============================
# IMPORTS
# ==============================
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from delta import *

# ==============================
# PATHS (AJUSTE CONFORME SEU AMBIENTE)
# ==============================
data_path = "data/vendas.csv"
delta_path = "data/delta/vendas"

# ==============================
# SPARK SESSION COM DELTA
# ==============================
spark = SparkSession.builder \
    .appName("DeltaAtividade") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Estrutura do Schema CSV


schema = StructType([
    StructField("id_venda", IntegerType(), True),
    StructField("produto", StringType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("valor_unitario", DoubleType(), True),
    StructField("data_venda", StringType(), True),
    StructField("regiao", StringType(), True)
])

In [ ]:
# Leitura CSV

df = spark.read.csv(
    data_path,
    header=True,
    schema=schema
)

df.show()
df.printSchema()

In [ ]:
# Escrita Delta lake

df.write.format("delta") \
    .mode("overwrite") \
    .save(delta_path)


In [ ]:
# Criação de tabela Delta Lake


spark.sql(f"""
CREATE TABLE IF NOT EXISTS vendas_delta (
    id_venda INT,
    produto STRING,
    quantidade INT,
    valor_unitario DOUBLE,
    data_venda STRING,
    regiao STRING
)
USING DELTA
LOCATION '{delta_path}'
""")


In [ ]:
# INSERT


spark.sql("""
INSERT INTO vendas_delta VALUES
(2000, 'Mouse Gamer', 2, 150.0, '2024-03-01', 'Sul'),
(2001, 'Teclado Mecânico', 1, 300.0, '2024-03-01', 'Sudeste')
""")

In [ ]:
# UPDDATE

spark.sql("""
UPDATE vendas_delta
SET valor_unitario = 999.0
WHERE id_venda = 2000
""")

In [ ]:
# DELETE

spark.sql("""
DELETE FROM vendas_delta
WHERE id_venda = 2001
""")


In [ ]:
# Consulta final para verificar alterações
 
spark.sql("SELECT * FROM vendas_delta").show()


In [ ]:
 # TIME TRAVEL, Histórico de versões

spark.sql("DESCRIBE HISTORY vendas_delta").show()


In [ ]:
 # TIME TRAVEL, Consulta versão anterior

spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path) \
    .show()